# Install Dependencies

In [1]:
!pip install openai numpy Pillow requests scikit-image

In [2]:
import openai
import numpy as np
from PIL import Image
import requests
from io import BytesIO
from skimage.metrics import structural_similarity as ssim
import base64
import imghdr

C:\Users\panka\AppData\Local\Temp\ipykernel_2228\2297944215.py:8: DeprecationWarning: 'imghdr' is deprecated and slated for removal in Python 3.13
  import imghdr


In [3]:
def simiarity_image(image_path1,image_path2):
   with open(image_path1, "rb") as f:
        img_bytes = f.read()
   img_b64_1 = base64.b64encode(img_bytes).decode("utf-8")

   with open(image_path2, "rb") as f:
        img_bytes = f.read()
   img_b64_2 = base64.b64encode(img_bytes).decode("utf-8")

    
    
    # Compare two images semantically
   response = openai.chat.completions.create(
        model="gpt-4.1",   # vision-capable model
        max_tokens=300,
        messages=[
            {"role": "system", "content": "You are an assistant that compares images."},
            {"role": "user", "content": [
                {"type": "text", "text": "Compare these two images and describe similarities and differences."},
                {"type": "image_url",
                 "image_url": {
                        "url": f"data:image/{imghdr.what(image_path1)};base64,{img_b64_1}"
                    }
                },
                {"type": "image_url",
                    "image_url": {
                        "url": f"data:image/{imghdr.what(image_path2)};base64,{img_b64_2}"
                    }
                }
            ]}
        ]
    )
   return response.choices[0].message.content

In [4]:
def image_display(img_url):
    img = Image(url=img_url)
    display(img)

# Example Usage

In [5]:
import cv2
from skimage.metrics import structural_similarity as ssim
from IPython.display import Image, display

imagefilepath1="findsimilarityimage1.jpg"
imagefilepath2="findsimilarityimage2.jpg"

# Load images in grayscale
img1 = cv2.imread(imagefilepath1, cv2.IMREAD_GRAYSCALE)
img2 = cv2.imread(imagefilepath2, cv2.IMREAD_GRAYSCALE)

# Resize to same shape
img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))

print("____________Image1_____________")
image_display(imagefilepath1)
print("____________Image2_____________")
image_display(imagefilepath2)
print("____________Similarity and Difference_____________")
print(simiarity_image(imagefilepath1,imagefilepath2))


# Compute SSIM
score, diff = ssim(img1, img2, full=True)
print(f"Structural Similarity Index: {score:.4f}")
diff = (diff * 255).astype("uint8")
print(f"SSIM score: {score:.4f}")

# Threshold the difference map
thresh = cv2.threshold(diff, 0, 255,
                       cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)[1]

# Find contours of differing regions
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL,
                               cv2.CHAIN_APPROX_SIMPLE)

# Collect difference points (bounding boxes)
difference_points = []
output = cv2.cvtColor(img1, cv2.COLOR_GRAY2BGR)
for c in contours:
    x, y, w, h = cv2.boundingRect(c)
    difference_points.append((x, y, w, h))
    cv2.rectangle(output, (x, y), (x+w, y+h), (0, 0, 255), 2)

# Save result with highlighted differences
cv2.imwrite("differences.png", output)
print("Points of difference (bounding boxes):", difference_points)

print("____________Differnce_____________")
image_display("differences.png")

____________Image1_____________


____________Image2_____________


____________Similarity and Difference_____________
Certainly! Here is a comparison of the two images:

**Similarities:**
- Both images feature the same man, standing in the same pose on the same residential street.
- The weather and lighting conditions appear identical in both images.
- The surroundings, including the houses, cars, trees, and hedges, are the same in both images.
- The camera angle and perspective are nearly identical.

**Differences:**
- The most notable difference is the shadow of the man:
  - In the **first image**, the man's shadow is long, narrow, and quite realistic.
  - In the **second image**, the man's shadow is shorter, broader, and has a cartoonish head, making it appear unnatural and altered.
- There is a green recycling bin with some white markings present behind the man and to the right in the **second image** that does not appear in the **first image**.
- The shadows cast by the cars and other elements seem consistent in both, suggesting the primary edit 